# Clean Multi-Provider RAG Version for your documents

This RAG is a multi-provider, document-based question answering system.
It allows users to upload a document and ask questions strictly grounded in that document.

The system does not persist any data. All indexing and retrieval exist only in memory during runtime.

#### Level Flow

- User uploads a document.
- The document is validated (size limit enforced).
- The document is chunked using controlled overlap.
- Chunks are embedded using the selected provider.
- Chunks are stored in an in-memory vector store.
- User asks a question.
- Top-k relevant chunks are retrieved using Max Marginal Relevance (MMR).
- The model generates an answer strictly based on retrieved context.
- Citations (chunk source references) are displayed.

#### Core Design Decisions
1. Multi-Provider Support

    The system supports:
    - OpenAI
    - OpenRouter
    - Ollama (local)

    Both LLM generation and embeddings are provider-aware and selected dynamically.

2. Ephemeral Memory Only

    - Vector store: in-memory
    - Document store in-memory
    - No disk persistence
    - No external storage

    This ensures uploaded documents are not retained after runtime ends.

3. Additional Safeguards

    - Document size limit
    - Token estimation before generation
    - Embedding caching in memory
    - Streaming responses
    - Citation display

In [53]:
# Setup
import os
import uuid
import hashlib
import tiktoken
import gradio as gr
from enum import Enum
from dotenv import load_dotenv
from PyPDF2 import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, ChatOllama

load_dotenv()

True

In [54]:
class Provider(str, Enum):
    OPENAI = "OpenAI"
    OPENROUTER = "OpenRouter"
    OLLAMA = "Ollama"


def get_llm(provider: Provider):
    if provider == Provider.OPENAI:
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.1,
            api_key=os.getenv("OPENAI_API_KEY"),
        )

    if provider == Provider.OPENROUTER:
        return ChatOpenAI(
            model="openai/gpt-4o-mini",
            temperature=0.1,
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
        )

    if provider == Provider.OLLAMA:
        return ChatOllama(
            model="llama3",
            temperature=0.1,
        )


def get_embeddings(provider: Provider):
    if provider == Provider.OPENAI:
        return OpenAIEmbeddings(
            model="text-embedding-3-small",
            api_key=os.getenv("OPENAI_API_KEY"),
        )

    if provider == Provider.OPENROUTER:
        return OpenAIEmbeddings(
            model="text-embedding-3-small",
            base_url="https://openrouter.ai/api/v1",
            api_key=os.getenv("OPENROUTER_API_KEY"),
        )

    if provider == Provider.OLLAMA:
        return OllamaEmbeddings(model="nomic-embed-text")

In [55]:
# doc size limit
MAX_DOC_SIZE_MB = 5
embedding_cache = {}
vectorstore = None

In [56]:
# Validate doc size
def validate_file(file_path):
    if not file_path:
        return "No file uploaded."

    size_mb = os.path.getsize(file_path) / (1024 * 1024)

    if size_mb > MAX_DOC_SIZE_MB:
        return f"File too large. Limit is {MAX_DOC_SIZE_MB} MB."

    return None

In [57]:
def load_document(file_path):
    ext = os.path.splitext(file_path)[1].lower()

    if ext in [".txt", ".md"]:
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text

    else:
        raise ValueError("Unsupported file type")

In [58]:
# Token estimation before sending to the model
def estimate_tokens(text: str, model="gpt-4o-mini"):
    try:
        enc = tiktoken.encoding_for_model(model)
        return len(enc.encode(text))
    except Exception:
        return len(text) // 4

In [59]:
# Embedding caching in memory
def get_cached_embedding(text, embeddings):
    key = hashlib.sha256(text.encode()).hexdigest()

    if key in embedding_cache:
        return embedding_cache[key]

    vector = embeddings.embed_query(text)
    embedding_cache[key] = vector
    return vector

In [60]:
# Chunking
def chunk_document(text: str, source: str):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150,
    )

    chunks = splitter.split_text(text)

    documents = []

    for i, chunk in enumerate(chunks):
        documents.append(
            Document(
                page_content=chunk,
                metadata={
                    "source": source,
                    "chunk_id": i,
                }
            )
        )

    return documents

In [61]:
# RAG pipeline (Ephemeral Only)
vectorstore = None
docstore = InMemoryStore()


def index_text(text: str, source: str, provider: Provider):
    global vectorstore

    embeddings = get_embeddings(provider)

    vectorstore = Chroma(
        collection_name="pchat",
        embedding_function=embeddings,
    )

    docs = chunk_document(text, source)

    vectorstore.add_documents(docs)

In [62]:
# Retrival with Max Marginal Relevance: This avoids redundant chunks
def retrieve(query: str, top_k: int):
    if vectorstore is None:
        return [], ""

    results = vectorstore.max_marginal_relevance_search(
        query,
        k=top_k,
        fetch_k=top_k * 3
    )

    context = "\n\n".join([doc.page_content for doc in results])

    return results, context

In [63]:
# QA
def stream_answer(query: str, provider, top_k):
    results, context = retrieve(query, top_k)

    if not context:
        yield "No document indexed."
        return

    llm = get_llm(provider)

    system_prompt = """
Answer strictly using the provided context.
If not found, say:
'I cannot find this information in the provided document.'
Be concise.
"""

    messages = [
        ("system", f"{system_prompt}\n\nContext:\n{context}"),
        ("user", query),
    ]

    full = ""

    for chunk in llm.stream(messages):
        full += chunk.content or ""
        yield full

    # Append citations
    sources = list(
        {f"{doc.metadata['source']} (chunk {doc.metadata['chunk_id']})"
         for doc in results}
    )

    citation_block = "\n\n---\nSources:\n" + "\n".join(sources)

    yield full + citation_block


In [64]:
# UI
with gr.Blocks(title="Multi-Provider RAG system") as ui:
    gr.Markdown("# Multi-Provider RAG system")

    provider = gr.Dropdown(
        choices=[p.value for p in Provider],
        value=Provider.OPENROUTER.value,
        label="Provider"
    )

    top_k = gr.Slider(1, 8, value=4, step=1, label="Top K Chunks")

    file_input = gr.File(
        type="filepath",
        file_types=[".txt", ".pdf", ".md"]
    )
    token_info = gr.Markdown()

    question = gr.Textbox(label="Ask a question")
    answer = gr.Markdown()

    def handle_upload(file, selected_provider):
        validate_file(file)
        text = load_document(file)

        index_text(text, os.path.basename(file), Provider(selected_provider))

        tokens = estimate_tokens(text)
        return f"Document indexed. Estimated tokens: {tokens}"

    file_input.upload(
        handle_upload,
        inputs=[file_input, provider],
        outputs=token_info
    )

    question.submit(
        stream_answer,
        inputs=[question, provider, top_k],
        outputs=answer
    )

ui.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
